# Seminar 4: Prompt Engineering

Our today's notebook is designed to replace a separate slide deck: the markdown cells contain the essential concepts, while the code cells let us immediately test them.

## Practical focus:

1. **Zero-shot, one-shot and few-shot prompting**
2. **Prompt templates**
3. **Context windows and context management**
4. **Chain-of-Thought (CoT) / reasoning-oriented prompting**

We will also use several elementary ideas throughout:

- clear task/instruction,
- relevant context,
- constraints,
- output format,
- structured output,
- evaluation and iteration.

## Seminar structure:

### Part A — Guided concepts (~45 min)
We build up the main techniques step by step.

### Part B — Practice (~45 min)
You modify, compare, test and evaluate prompts yourself.

> **Main idea:** Prompt engineering is not about finding a magic phrase. It is about making the task, context, constraints and expected output explicit — and then testing whether the prompt works.

# 0. Setup

The notebook uses the **Groq API**.

Enter your API key when prompted. `getpass()` hides the key so it is not displayed in the notebook.

Link to Groq: https://console.groq.com/keys

In [ ]:
from groq import Groq
from getpass import getpass
import json

#API_KEY = getpass("Enter your Groq API key: ")
MODEL = "openai/gpt-oss-120b"

client = Groq(api_key="")

def ask_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=1000
    )

    return response.choices[0].message.content

print("Setup complete!")

Setup complete!


---
# PART A — MAIN PROMPT ENGINEERING TECHNIQUES
## Guided discussion and demonstrations

# 1. Start with the anatomy of a good prompt

Before learning named techniques, start with four elementary building blocks:

### 1. Task/instruction
What exactly should the model do?

### 2. Context/input
What information does the model need?

### 3. Constraints
What is allowed or forbidden? What rules should it follow?

### 4. Output format
What should the answer look like?

Real-world prompts often combine these components. Examples include financial assistants with contextual variables, classifiers with tightly defined labels, and production prompts that explicitly specify JSON schemas.

### Example: an underspecified prompt

In [32]:
prompt = """
Classify this customer message:

My password reset link isn't working.
"""

print(ask_llm(prompt))

Password Reset Issue


## Why is the prompt underspecified?

The model has to guess:

- what the possible categories are
- how ambiguous cases should be handled
- what format to return

For example, a broken password-reset link could plausibly be interpreted as either **Account Access** or a **Technical Issue** unless we define our classification policy.

A more explicit version:

In [33]:
prompt = """
TASK:
Classify the customer message.

ALLOWED CATEGORIES:
- Billing
- Technical Issue
- Account Access

CLASSIFICATION RULE:
Problems involving login, passwords, or password resets belong to Account Access.

CUSTOMER MESSAGE:
My password reset link isn't working.

CONSTRAINT:
Choose exactly one allowed category.

OUTPUT:
Return only the category name.
"""

print(ask_llm(prompt))

KeyboardInterrupt: 

### Important lesson

A model cannot infer your organization's hidden business rules reliably.

Prompt engineering often means **turning implicit expectations into explicit instructions**.

# 2. Zero-shot, One-shot and Few-shot Prompting

Examples inside a prompt can demonstrate the task.

### Zero-shot
The model receives instructions but **no demonstration**.

### One-shot
The model receives **one example**.

### Few-shot
The model receives **several examples**.

This is called **in-context learning**: the examples are provided in the current prompt/context rather than used to retrain the model.

Examples are particularly useful when:

- labels have specialized meanings
- the desired format is unusual
- the distinction between classes is difficult to describe
- demonstrating the task is easier than explaining every rule.

But examples also consume context and can introduce bias if they are poor or unrepresentative.

In [ ]:
ticket = "Payment succeeded but I still cannot access my account."

zero_shot = f"""
Classify as Billing, Technical Issue, or Account Access.

Ticket: {ticket}

Return only the category.
"""

one_shot = f"""
Classify as Billing, Technical Issue, or Account Access.

EXAMPLE:
Ticket: I forgot my password.
Category: Account Access

NOW CLASSIFY:
Ticket: {ticket}
Category:
"""

few_shot = f"""
Classify as Billing, Technical Issue, or Account Access.

EXAMPLES:
Ticket: I was charged twice.
Category: Billing

Ticket: The application crashes.
Category: Technical Issue

Ticket: I forgot my password.
Category: Account Access

NOW CLASSIFY:
Ticket: {ticket}
Category:
"""

print("ZERO-SHOT:", ask_llm(zero_shot))
print("ONE-SHOT:", ask_llm(one_shot))
print("FEW-SHOT:", ask_llm(few_shot))

ZERO-SHOT: Account Access
ONE-SHOT: Account Access
FEW-SHOT: Account Access


## What should we notice?

Few-shot prompting is **not automatically better**.

Ask yourself:

- Are the examples representative?
- Do they clarify difficult boundaries?
- Could they bias the model?
- Is the extra context worth the additional tokens?

Production prompts often include both **positive and negative examples** when the boundary between classes matters. This pattern also appears in real-world classifier prompts.

# 3. Prompt Templates

In an application, the instructions often remain the same while the input changes.

A **prompt template** separates:

- the fixed instructions
- the variable input.

Instead of manually rewriting:

`Classify "I forgot my password"...`

and later:

`Classify "I was charged twice"...`

we create one reusable template.

The `{ticket}` part of an f-string is replaced with the current value of the variable.

In [ ]:
def make_prompt(ticket):
    prompt = f"""
TASK:
Classify this support ticket.

ALLOWED CATEGORIES:
- Billing
- Technical Issue
- Account Access

TICKET:
{ticket}

OUTPUT:
Return only the category.
"""
    return prompt


ticket = "I forgot my password."

prompt = make_prompt(ticket)

print("THE COMPLETED PROMPT:")
print(prompt)

print("\nMODEL ANSWER:")
print(ask_llm(prompt))

THE COMPLETED PROMPT:

TASK:
Classify this support ticket.

ALLOWED CATEGORIES:
- Billing
- Technical Issue
- Account Access

TICKET:
I forgot my password.

OUTPUT:
Return only the category.


MODEL ANSWER:
Account Access


## Why are templates important?

They make prompts:

- **reusable** — the same instructions work with many inputs
- **consistent** — every input receives the same rules
- **maintainable** — change the template once rather than many copies
- **testable** — run the same prompt design across a dataset

Many real production prompts are templates containing placeholders such as `{question}`, `{table_schemas}`, `{location}` or `{strategic_prompt}`.

> A prompt template is not an advanced framework feature. At its simplest, it is just reusable text with variables.

# 4. Context Windows, Tokens and Context Management

An LLM can only process a finite amount of information at once. This is its **context window**.

The context can include:

- instructions
- conversation history
- examples
- retrieved documents
- tool results
- the user's current question
- and generated output, depending on the API/model.

Models process text as **tokens**, not directly as words.

A token can be:

- a whole word
- part of a word
- punctuation
- whitespace or another text fragment.

Therefore:

> **1 word ≠ necessarily 1 token**

Longer context can increase cost and latency and can also make it harder for the model to focus on relevant information.

## 4.1 Tokenizer demonstration

`tiktoken` lets us inspect tokenization: https://github.com/openai/tiktoken

**Note:** this is an approximate demonstration of tokenization. Different model families/providers can use different tokenizers, so do not interpret these counts as exact Groq billing counts!

In [ ]:
# Run once if tiktoken is not installed:
# !pip install -q tiktoken

import tiktoken

tokenizer = tiktoken.get_encoding("o200k_base")

text = "I cannot log into my account."

tokens = tokenizer.encode(text)

print("Text:", text)
print("Words (roughly):", len(text.split()))
print("Tokens:", len(tokens))
print("Token IDs:", tokens)

Text: I cannot log into my account.
Words (roughly): 6
Tokens: 7
Token IDs: [40, 6284, 2142, 1511, 922, 3527, 13]


In [ ]:
print("Individual token pieces:")

for token in tokens:
    print(repr(tokenizer.decode([token])))

Individual token pieces:
'I'
' cannot'
' log'
' into'
' my'
' account'
'.'


## 4.2 Prompt length consumes context

Compare a short prompt and a prompt containing several examples.

In [ ]:
short_prompt = """
Classify: I cannot log in.
"""

long_prompt = """
Classify as Billing, Technical Issue, or Account Access.

Example: I was charged twice. -> Billing
Example: The application crashes. -> Technical Issue
Example: I forgot my password. -> Account Access
Example: My reset link expired. -> Account Access

Now classify:
I cannot log in.
"""

print("Short prompt:", len(tokenizer.encode(short_prompt)), "tokens")
print("Long prompt:", len(tokenizer.encode(long_prompt)), "tokens")

Short prompt: 9 tokens
Long prompt: 63 tokens


## 4.3 Context management

When the available information becomes too large, we need to decide **what deserves space in the context window**.

Four common strategies:

### Trim
Remove irrelevant or old information.

### Summarize
Compress older conversation or documents.

### Retrieve
Include only information relevant to the current question. This is a key idea behind **RAG**.

### Prioritize
Keep important instructions and relevant evidence; remove redundant examples or boilerplate.

### The trade-off

More context is not automatically better.

Useful context should improve the task enough to justify:

- additional tokens
- additional cost
- additional latency
- and possible distraction.

This connects directly to few-shot prompting: every example also occupies context.

# 5. Chain-of-Thought (CoT) and Reasoning-Oriented Prompting

For tasks requiring several reasoning steps, a prompt can ask the model to work through intermediate steps before producing an answer.

This family of techniques is commonly called **Chain-of-Thought (CoT) prompting**.

A classic distinction is:

### Direct prompting
Ask only for the answer.

### Reasoning-oriented prompting
Ask the model to perform intermediate reasoning/calculation before answering.

For classroom purposes, we will request a **brief calculation or concise explanation** rather than an unnecessarily long reasoning trace.

In [ ]:
problem = """
A company invests €1,000 at 5% simple annual interest for 2 years.
What is the final amount?
"""

direct_prompt = f"""
Solve the problem.
Return only the final answer.

{problem}
"""

reasoning_prompt = f"""
Solve the problem.
Show a brief calculation, then give the final answer.

{problem}
"""

print("DIRECT ANSWER:")
print(ask_llm(direct_prompt))

print("\nWITH BRIEF REASONING:")
print(ask_llm(reasoning_prompt))

DIRECT ANSWER:
€1,100.

WITH BRIEF REASONING:
**Brief calculation**

Simple interest = Principal × rate × time  
\(= €1{,}000 \times 0.05 \times 2 = €100\)

Final amount = Principal + interest  
\(= €1{,}000 + €100 = €1{,}100\)

**Final answer:** €1,100.


## When can reasoning-oriented prompting help?

Potentially useful for:

- multi-step calculations
- logical problems
- planning
- tasks where intermediate checks matter.

Less useful for:

- simple classification
- extraction
- straightforward formatting
- tasks where the extra explanation adds cost without improving quality.

### Important principle

> **Match the prompting technique to the task.**

Do not add CoT, few-shot examples or long instructions simply because they sound more sophisticated.

# 6. Structured Output and Constraints

Another elementary but highly practical technique is to specify a machine-readable output format.

A human may understand:

> “This looks like an urgent account access issue.”

Software often needs something predictable such as JSON.

In [ ]:
ticket = "My password reset link expired and I cannot log in."

prompt = f"""
TASK:
Analyze the customer support message.

ALLOWED CATEGORIES:
- Billing
- Technical Issue
- Account Access

ALLOWED URGENCY:
- low
- medium
- high

CUSTOMER MESSAGE:
{ticket}

OUTPUT:
Return ONLY valid JSON:
{{
  "category": "...",
  "urgency": "...",
  "confidence": 0.0
}}

CONSTRAINTS:
- Confidence must be between 0 and 1.
- Do not include text before or after the JSON.
"""

answer = ask_llm(prompt)
print(answer)

{
  "category": "Account Access",
  "urgency": "medium",
  "confidence": 0.97
}


## Validation happens outside the model

Asking for JSON does not guarantee valid JSON.

The application can check the response **without making another API call**.

In [ ]:
try:
    data = json.loads(answer)

    required = ["category", "urgency", "confidence"]
    fields_ok = all(field in data for field in required)

    if fields_ok:
        print("Readable JSON with all required fields.")
        print(data)
    else:
        print("JSON is readable, but a required field is missing.")

except json.JSONDecodeError:
    print("The model did not return valid JSON.")

Readable JSON with all required fields.
{'category': 'Account Access', 'urgency': 'medium', 'confidence': 0.97}


# 7. Prompt Engineering Is an Experimental Process

There is rarely one universally “best” prompt.

A useful workflow is:

> **Design → Test → Evaluate → Inspect failures → Refine → Test again**

Possible evaluation criteria:

- task accuracy
- format validity
- robustness on ambiguous cases
- robustness on adversarial cases
- token usage/cost
- latency
- consistency.

### First takeaway

A stronger prompt is usually not the one with the most words.

It is the one that gives the model the **right information and constraints for the task**.

---
# PART B — PRACTICAL EXERCISES

Now use the techniques rather than only reading about them.

The exercises deliberately reuse one customer support setting so that you can focus on **prompt design**, not on learning a new domain each time.

# Exercise 1 — Repair an underspecified prompt

Start with:

> `Classify this customer message: "My password reset link isn't working."`

### Your task

Rewrite it so that:

1. the task is explicit
2. categories are defined
3. ambiguity is reduced
4. the output format is specified.

Then run both prompts and compare the outputs.

In [ ]:
ticket="I want to order some black coffee, please"

bad_prompt = f"""
Classify this customer message:
{ticket}
"""

your_prompt = f"""
Classify this customer message: {ticket}
Categories: ORDER, INQUIRY, COMPLAINT, CANCELLATION

OUTPUT:
return exactly one category.
"""

print("ORIGINAL:")
print(ask_llm(bad_prompt))

print("\nYOUR VERSION:")
print(ask_llm(your_prompt))

ORIGINAL:
**Intent:** Order (request to place an order for black coffee)

YOUR VERSION:
ORDER


# Exercise 2 — Zero-shot vs Few-shot

Use this ambiguous ticket:

> `Payment succeeded but I still cannot access my account.`

### Your task

Create:

1. a zero-shot prompt
2. a few-shot prompt with three useful examples.

Before running them, predict whether the answers will differ.

Then answer:

- Did examples help?
- Were your examples representative?
- How many extra tokens did the examples consume?

In [41]:
ticket = "Payment succeeded but I still cannot access my account."

zero_shot = f"""
Classify this customer message: {ticket}
Categories: ORDER, INQUIRY, COMPLAINT, CANCELLATION

OUTPUT:
return exactly one category.
"""

few_shot = f"""
Classify this customer message: {ticket}
Categories: ORDER, INQUIRY, COMPLAINT, CANCELLATION

examples:

Ticket: I want to order coffee.
Category: ORDER

Ticket: I cannot complete the payment, I want to cancel.
Category: CANCELLATION

Ticket: This is unacceptable service, I want free stuff.
Category: COMPLAINT

OUTPUT:
return exactly one category.
"""

print("ZERO-SHOT:")
print(ask_llm(zero_shot))

print("\nFEW-SHOT:")
print(ask_llm(few_shot))

print("\nTOKEN COUNTS:")
print("Zero-shot:", len(tokenizer.encode(zero_shot)))
print("Few-shot:", len(tokenizer.encode(few_shot)))

ZERO-SHOT:
COMPLAINT

FEW-SHOT:
COMPLAINT

TOKEN COUNTS:
Zero-shot: 41
Few-shot: 93


# Exercise 3 — Build and reuse a prompt template

### Your task

Complete `make_prompt()` once, then use it for all three tickets.

The **instructions should remain the same** while only the ticket changes.

In [42]:
def make_prompt(ticket):
    return f"""
Classify this customer message: {ticket}
Categories: ORDER, INQUIRY, COMPLAINT, CANCELLATION

OUTPUT:
return exactly one category.
"""


tickets = [
    "I forgot my password.",
    "I was charged twice.",
    "The application crashes when I upload a file."
]

for ticket in tickets:
    prompt = make_prompt(ticket)

    print("\nTICKET:", ticket)
    print("ANSWER:", ask_llm(prompt))


TICKET: I forgot my password.
ANSWER: INQUIRY

TICKET: I was charged twice.
ANSWER: COMPLAINT

TICKET: The application crashes when I upload a file.
ANSWER: COMPLAINT


# Exercise 4 — Context-window challenge

Suppose you have limited context space.

Which information should you keep?

### Scenario

You are classifying a support ticket. Your prompt contains:

- 20 old conversation messages
- 10 classification examples
- the category definitions
- a long company history
- the current ticket
- output-format instructions.

### Your task

Rank these from **most important to least important** for the current classification task.

Then modify the text below and compare token counts before and after your reduction.

In [43]:
large_context = """
CATEGORY DEFINITIONS:
Billing = payment, invoice and refund problems.
Technical Issue = application or software malfunction.
Account Access = login, password and authentication problems.

OLD CONVERSATION:
Customer previously asked about opening hours.
Customer asked about the mobile app.
Customer thanked the support agent.
Customer discussed an unrelated product.
Customer asked another unrelated question.

COMPANY HISTORY:
Our company was founded many years ago and has expanded into several markets.
It has offices in multiple countries and offers many different products.

EXAMPLES:
I was charged twice. -> Billing
The app crashes. -> Technical Issue
I forgot my password. -> Account Access
My refund is missing. -> Billing
My reset link expired. -> Account Access

CURRENT TICKET:
I cannot sign into my account.

OUTPUT:
Return only the category.
"""

reduced_context = """
CATEGORY DEFINITIONS:
Billing = payment, invoice and refund problems.
Technical Issue = application or software malfunction.
Account Access = login, password and authentication problems.


EXAMPLES:
I was charged twice. -> Billing
The app crashes. -> Technical Issue
I forgot my password. -> Account Access
My refund is missing. -> Billing
My reset link expired. -> Account Access

CURRENT TICKET:
I cannot sign into my account.

OUTPUT:
Return only the category.
"""

print("Original:", len(tokenizer.encode(large_context)), "tokens")
print("Reduced:", len(tokenizer.encode(reduced_context)), "tokens")

Original: 162 tokens
Reduced: 95 tokens


# Exercise 5 — Does CoT help?

Test two prompting styles on a problem that requires several steps.

### Your task

1. Write a direct answer prompt.
2. Write a reasoning-oriented prompt requesting a brief calculation.
3. Compare the answers.
4. Compare token usage.
5. Decide whether the extra reasoning was useful.

In [45]:
problem = """
A shop buys 20 products for €12 each.
It sells 15 of them for €18 each and the remaining 5 for €10 each.
What is the total profit?
"""

direct_prompt = f"""
SOLVE THIS PROBLEM, GIVE BACK THE ANSWER ONLY.

{problem}
"""

cot_prompt = f"""
SOLVE THE PROBLEM BELOW, AND SHOW THE PROCESS HOW YOU SOLVED IT. WRITE SHORT EXPLAINATION. ONLY PUT THE STEPS.

{problem}
"""

direct_answer = ask_llm(direct_prompt)
cot_answer = ask_llm(cot_prompt)

print("DIRECT:")
print(direct_answer)

print("\nCOT / REASONING-ORIENTED:")
print(cot_answer)

print("\nPROMPT TOKENS:")
print("Direct:", len(tokenizer.encode(direct_prompt)))
print("CoT:", len(tokenizer.encode(cot_prompt)))

DIRECT:
€80

COT / REASONING-ORIENTED:
**Steps**

1. Compute total cost:  
   \(20 \text{ products} \times €12 = €240\)

2. Compute revenue from the first batch:  
   \(15 \text{ products} \times €18 = €270\)

3. Compute revenue from the second batch:  
   \(5 \text{ products} \times €10 = €50\)

4. Compute total revenue:  
   \(€270 + €50 = €320\)

5. Compute profit:  
   \(€320\;(\text{revenue}) - €240\;(\text{cost}) = €80\)

**Total profit = €80**

PROMPT TOKENS:
Direct: 51
CoT: 68


# Exercise 6 — Final integration challenge

Build one reusable customer support prompt that combines today's ideas.

It must:

1. classify the message
2. determine urgency
3. generate a short helpful response
4. return valid JSON
5. use a reusable template.

### Required output

```json
{
  "category": "...",
  "urgency": "...",
  "response": "..."
}
```

### Test it on at least three different messages

Ask yourself:

- Did you need few-shot examples?
- How much context did they add?
- Is every instruction necessary?
- Is the JSON valid?

In [49]:
def make_support_prompt(customer_message):
    return f"""
CLASSIFY THIS TICKET.

CATEGORIES: ORDER - product orders, INQUIRY - questions about products, COMPLAINT - problems not resulting on cancellation, CANCELLATION - cancellation requests

URGENCY: 1 -  low, 2 - normal, 3 - high

CONFIDENCE: 0.00 to 1.00 - how confident is your pediction

REQUIRED FORMAT, JSON:
(category: '', urgency: 0, confidence: 0.00)

EXAMPLES

Ticket: "I want to order coffee"
Category: ORDER
URGENCY: 2

Ticket: "I want to cancel my order, the product is good, but i dont need it"
Category: CANCELLATION
URGENCY: 1

Ticket: "I don't like your product, you should close your shop."
Category: COMPLAINT
URGENCY: 3

Ticket: "I want to ask about your coffee, if it is sweet or not."
Category: INQUIRY
URGENCY: 2

CUSTOMER MESSAGE:
{customer_message}
"""


test_messages = [
    "I was charged twice for my subscription.",
    "My password reset link expired.",
    "The application crashes whenever I upload a file."
]

for message in test_messages:
    prompt = make_support_prompt(message)
    answer = ask_llm(prompt)

    print("\nMESSAGE:", message)
    print("ANSWER:", answer)


MESSAGE: I was charged twice for my subscription.
ANSWER: {
  "category": "COMPLAINT",
  "urgency": 3,
  "confidence": 0.98
}

MESSAGE: My password reset link expired.
ANSWER: {
  "category": "INQUIRY",
  "urgency": 2,
  "confidence": 0.95
}

MESSAGE: The application crashes whenever I upload a file.
ANSWER: {
  "category": "COMPLAINT",
  "urgency": 3,
  "confidence": 0.98
}


# Seminar summary

## Four mandatory techniques:

- **Zero-/one-/few-shot prompting:** demonstrate the task through examples.
- **Prompt templates:** separate reusable instructions from changing input.
- **Context management:** decide what information deserves limited context-window space.
- **CoT / reasoning-oriented prompting:** use intermediate reasoning selectively for tasks that benefit from it.

## Elementary techniques that support all four:

- explicit task
- relevant context
- constraints
- output format
- structured output
- evaluation and iteration.

## Final principle:

> **Use the simplest prompt that performs the task reliably. Add complexity only when testing shows that it helps.**